# Inspect ChromaDB Chunks

This notebook loads your local ChromaDB collection and prints chunk text + metadata.


In [80]:
import chromadb
import pandas as pd
from pathlib import Path
import json

# Use the same Chroma persist dir as the backend.
PROJECT_ROOT = Path("/Users/ajeyds/Projects/RAG")
PERSIST_DIR = str(PROJECT_ROOT / "data/chroma_db")
COLLECTION_NAME = "rag_chunks"

# Filter to a single uploaded file.
# Default: auto-pick the latest entry from data/kb_metadata.json.
SOURCE_PATH = None

try:
    meta_path = PROJECT_ROOT / "data/kb_metadata.json"
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    latest_entry = max(meta["files"].values(), key=lambda e: e.get("uploadedAt", ""))
    SOURCE_PATH = latest_entry.get("path")
except Exception as e:
    print("Could not auto-resolve SOURCE_PATH:", e)


In [2]:
# 1) Connect to ChromaDB
client = chromadb.PersistentClient(path=PERSIST_DIR)
collection = client.get_collection(name=COLLECTION_NAME)

print("Notebook cwd:", Path.cwd())
print("Using PERSIST_DIR:", PERSIST_DIR)
print("Collection:", COLLECTION_NAME)
print("Total items:", collection.count())


NameError: name 'chromadb' is not defined

In [1]:
# 2) Retrieve data
# NOTE: We intentionally do NOT include embeddings to keep the output small.
include = ["documents", "metadatas"]

if SOURCE_PATH:
    data = collection.get(where={"source": SOURCE_PATH}, include=include)
else:
    # Pull only the first N for quick inspection.
    # If you want everything, increase or remove the limit.
    data = collection.get(include=include, limit=200)

ids = data.get("ids", [])
documents = data.get("documents", [])
metadatas = data.get("metadatas", [])

print("Fetched rows:", len(ids))
print("Example metadata keys:", sorted((metadatas[0] or {}).keys()) if metadatas else [])


NameError: name 'SOURCE_PATH' is not defined

In [51]:
# 3) Format into a table
rows = []
for _id, doc, meta in zip(ids, documents, metadatas):
    meta = meta or {}
    rows.append(
        {
            "id": _id,
            "chunk_type": meta.get("chunk_type"),
            "chunk_id": meta.get("chunk_id"),
            "parent_context": meta.get("parent_context"),
            "source": meta.get("source"),
            "document": doc,
        }
    )

df = pd.DataFrame(rows)
df = df.sort_values(by=["chunk_type", "chunk_id"], na_position="last")

# Show a preview (no embeddings)
df.head(30)


KeyError: 'chunk_type'

In [40]:
# 4) Print chunk text (optionally truncate)
N = 10
max_chars = 800  # set to None for full text

for i, row in df.head(N).iterrows():
    print("\n" + "=" * 80)
    print(f"[{row['chunk_type']}] chunk_id={row['chunk_id']}")
    print(f"parent_context={row['parent_context']}")
    print(f"source={row['source']}")
    text = row["document"] or ""
    if max_chars is not None:
        text = text[:max_chars] + ("..." if len(text) > max_chars else "")
    print(text)



[ac] chunk_id=AC-1.1
parent_context=Vendor Master Synchronization
source=/Users/ajeyds/Projects/RAG/data/uploads/kb/c7b58460-0552-4804-be38-5222685d9732_rise2.docx
SAP Sync Execution — Given the SAP RFC interface is active, when the hourly job runs, then vendor data shall be fetched and updated in InLumin, and duplicates shall be prevented, and sync logs shall be stored.

[ac] chunk_id=AC-1.2
parent_context=Vendor Master Synchronization
source=/Users/ajeyds/Projects/RAG/data/uploads/kb/c7b58460-0552-4804-be38-5222685d9732_rise2.docx
Mandatory Field Validation — Given vendor data is received, when mandatory fields are missing, then the record shall be rejected and logged with error status.

[ac] chunk_id=AC-1.3
parent_context=Vendor Master Synchronization
source=/Users/ajeyds/Projects/RAG/data/uploads/kb/c7b58460-0552-4804-be38-5222685d9732_rise2.docx
Unique Constraint — Given Vendor Code + Company combination exists, when a duplicate is received, then the system shall update existing 